## Example Quadtree

First we load in all the required modules and python packages

In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from datetime import datetime
from pathlib import Path

from hydromt_sfincs import SfincsModel

from hydromt._utils import log

import hydromt_sfincs
import hydromt

# Initialize logging, the lower the log level number, the more verbose (more info) the output
# NOTSET=0-9, DEBUG=10, INFO=20, WARNING=30, ERROR=40, CRITICAL=50

log.initialize_logging()
log.set_log_level(log_level=20)

Check version of hydromt_sfincs, if you do not have these version this notebook will not work!

In [ ]:
# NOTE you need at least version >=1.3.0-rc5
print("HydroMT core version", hydromt.__version__)

# NOTE you need at least version >=2.0.0
print("HydroMT-SFINCS version", hydromt_sfincs.__version__)

### Regular grid model

In [ ]:
# Initialize SfincsModel Python class with the artifact data catalog which contains publically available data for North Italy
model_root = r"./regular_grid_model"

sf = SfincsModel(
    data_libs=["artifact_data"],  # specify which data libraries to use
    root=model_root,  # specify the root directory for the model
    mode="w+",  # specify the mode for opening the model (r=read only, r+=append, w=write, w+=overwrite
    write_gis=True,  # specify whether to write GIS data    ""
    )

In [ ]:
sf.grid.create(
    x0=318650,
    y0=5040000,
    dx=50.0,
    dy=50.0,
    nmax=107,
    mmax=250,
    rotation=27,
    epsg=32633,
)

In [ ]:
sf.write()

# Quadtree grid model

The procedure for building a quadtree grid is very similar to a regular grid. First we initialize a model again with a model root, data libraries and the mode specified.

In [ ]:
model_root = r"./quadtree_grid_model_with_snapwave_v05_igwaves_wavemaker"


In [ ]:
log_file = Path(model_root) / "hydromt_sfincs.log"

# Explicitly open log file for the session
log.initialize_logging()
logger = log._add_filehandler(log_file, log_level=20)

sf_qt = SfincsModel(
    data_libs=["artifact_data"],  # specify which data libraries to use
    root=model_root,  # specify the root directory for the model
    mode="w+",  # specify the mode for opening the model (r=read only, r+=append, w=write, w+=overwrite
    write_gis=True,  # specify whether to write GIS data    ""
)

To build a quadtree model you need to define an area that you would like to refine, for example along the coast, along a river or just any other area of interest. For this quadtree grid we will do a simple refinement along the coastline. We will do this based on a buffer of 500m along the OpenStreetMap coastline, which was downlaoded for you around Venice and part of the artifact data-catalog. In case you want to model another, the data can be downloaded [here](https://osmdata.openstreetmap.de/data/).

In [ ]:
# get OSM coastline data from data catalog within model bounding box of the regular grid model
gdf_osm = sf_qt.data_catalog.get_geodataframe("osm_coastlines", bbox=sf.bbox)

# convert polygon to line
gdf_osm_line = gdf_osm.to_crs(sf.crs).boundary

# add a buffer to the line of 500m
gdf_osm_buf = gdf_osm_line.buffer(500)

Make a plot of the regular model, with the coastline buffer


In [ ]:
# plot basemap of regular model
fig, ax = sf.plot_basemap(variable="dep", bmap="sat") # Original model

# plot buffered coastline on top
gdf_osm_buf.plot(ax=ax, color="purple", linewidth=0.5, alpha=0.9)

### Generate a Quadtree grid

This cell covers the most important step in a quadtree generation. As you can see, we are calling sf_qt.**quadtree_grid**.create(). This quadtree_grid component allows for defining a **refinement_polygon**, this refinement polygon should be a geodataframe that indicated both the geometry and refinement level of the refinement polygon. You can add a refinement level as indicated below. You could also add a "refinement_level" column to your existing geodataframe. 

A refinement level of 1, indicates that the grid will be refined (for this polygon) to half of its original resolution. For example:

| Refinement Level | Resolution |
|------------------|------------|
| Base Model       | 100 m      |
| Level 1          | 50 m       |
| Level 2          | 25 m       |
| Level 3          | 12.5 m     |

You can add polygons with multiple levels by adding to the list.

In [ ]:
# merge the buffered coastlines into one geometry and add a coordinate reference system
gdf_refinement = gpd.GeoDataFrame(
    geometry=[gdf_osm_buf.union_all()],
    # geometry=[gdf_osm_buf.union_all(), gdf_number_02.union_all()], #> in case of multiple polygons for refinement

    crs=sf.crs,
)
# set refinement level
# gdf_refinement["refinement_level"] = 1  # refinement level 1, refines once
gdf_refinement["refinement_level"] = 2  # refinement level 2, refines twice
# gdf_refinement["refinement_level"] = [1,3]  # e.g. refine once for the first polygon, 3 times for the second

# print the geodataframe
gdf_refinement


In [ ]:
# create quadtree grid with refinement polygons
sf_qt.quadtree_grid.create(
    x0=318650,
    y0=5040000,
    dx=50.0,
    dy=50.0,
    nmax=107,
    mmax=250,
    rotation=27,
    epsg=32633,
    refinement_polygons=gdf_refinement, # This is where we pass the refinement polygons, if not included, a quadtree model without refinement is created
)

In [ ]:
# Create figure and axes
fig, ax1 = sf.plot_basemap(variable="grid",plot_bounds=False, bmap="sat", figsize=(8, 5), zoomlevel = 12)

# Add nice titles and labels
ax1.set_title("Full Quadtree Grid", fontsize=14)
ax1.set_xlabel("Easting (m)")
ax1.set_ylabel("Northing (m)")

ax1.set_aspect("equal")

ax1.legend(loc="upper left")

# Clean layout and show
plt.tight_layout()
plt.show()

You can write the mesh away as geopackage, to load into Qgis

In [ ]:
import os
if not os.path.exists(os.path.join(sf_qt.root.path, "grid.gpkg")):

    gdf = sf_qt.quadtree_grid.data.level.ugrid.to_geodataframe()
    gdf.to_file(os.path.join(sf_qt.root.path, "grid.gpkg"))

Add a topobathy to the Quadtree grid, this is vary similar to the original scripts for regeular grid. For plotting the quadtree basemap, you can use sf_qt.plot_basemap(). However, plotting the bounds is not possible yet. Therefor you should use **plot_bounds = False**.

In [ ]:
elevation_list = [{"elevation": "merit_hydro", "zmin": 0.001}, {"elevation": "gebco"}]

sf_qt.quadtree_elevation.create(elevation_list=elevation_list)

sf_qt.plot_basemap(variable="z", plot_region=True, plot_bounds=False, bmap="sat")

Continue with the mask for the QuadTree grid; we aim to have the same active extent as the regular grid, therefore we use the region of the regular model as an include_polygon for the quadtree mask. Of course, we could also start from scratch and base the mask and its boundaries on a combination of polygons and elevation.

In [ ]:
# Method 1: using create methods
sf_qt.quadtree_mask.create_active(
    include_polygon=sf_qt.region,
)
sf_qt.quadtree_mask.create_boundary(
    btype="waterlevel",
    zmax=-4,
)

In [ ]:
sf_qt.plot_basemap(variable="mask", plot_bounds=False, bmap="sat", zoomlevel=12)

### Generate a SnapWave mask

By default you make a mask for SFINCS on your quadtree grid, but by specifying model='snapwave', you can make your SnapWave mask with using exactly the same function.

You can make the mask exactly the same as your SFINCS mask, but also make it different if needed.

And for the boundary cells, you then have to specify btype='waves'.

In [ ]:
# Method 1: using create methods
model = 'snapwave'
sf_qt.quadtree_mask.create_active(model=model,
    include_polygon=sf_qt.region,
)

sf_qt.quadtree_mask.create_boundary(
    model=model,
    btype="waves",
    zmax=-4,
    connectivity=4,
)

In [ ]:
# plot the snapwave mask
sf_qt.plot_basemap(variable="snapwave_mask", plot_bounds=False, bmap="sat", zoomlevel=12)

### Cut inactive cells

When you are happy with your mask, we can cut the inactive cells from the model. This is now possible because of the nature of the unstructured grids. By doing so, further pre-proccesing steps don't take into account the "inactive cells", and also the SFINCS model does not produce outputs anymore for these cells which keeps the output limited.

In [ ]:
sf_qt.quadtree_grid.cut_inactive_cells()

### Add an observation point

In [ ]:
x = 323653
y = 5044344
  
sf_qt.observation_points.add_point(x=x, y=y, name="station_01")

### Now add a GTSM waterlevel timeseries

For now, we add a simple waterlevel forcing based on GTSM reanalysis data. Since the data provided in the artificat data only covers a small period in time (and in space), it is important to first change the start and stop times of the model. 

NOTE, we already found a small little bug. To overcome this issue, we rename index coord into stations to match SFINCS convention. we a have made a issue https://github.com/Deltares/hydromt_sfincs/issues/303

In [ ]:
# Change period of model simulation time, specified in yyyymmdd HHMMSS, to match with the available water level data
sf_qt.config.update(
    {
        "tref": datetime(2010, 2, 5),
        "tstart": datetime(2010, 2, 5),
        "tstop": datetime(2010, 2, 5, 6),

    }
)

In [ ]:
sf_qt.water_level.create("gtsmv3_eu_era5", buffer=3000, merge=False)

# # NOTE, we already found a small little bug. To overcome this issue, we rename index coord into stations to match SFINCS convention.
# sf_qt.water_level._data = sf_qt.water_level.data.rename({'index': 'stations', "bzs": zs})
# or
sf_qt.config.update({"netbndbzsbzifile": None, "bndfile": "sfincs.bnd", "bzsfile": "sfincs.bzs"})

In [ ]:
sf_qt.config.update(
    **{
        "zsini": 0.25, #=initial water level > tricky for this case because the land is so low
        "dtmapout": 120, # map output every 120 seconds
        "dthisout": 10, # observation output every 10 seconds

    }
)

### Add SnapWave forcing of wave parameters

INFO > the class to build the netcdf wave boundary conditions input data is still in a branch.
It is easiest to force the wave boundary conditions using ascii input files like this for now:

'snapwave_bndfile = snapwave.bnd' in sfincs.inp > is exactly the same as the normal SFINCS bndfile, but then with wave boundary input locations:

        323664.8   5044037.0
        318671.9   5043342.7

'snapwave_bhsfile = snapwave.bhs' in sfincs.inp  > ascii file with wave significant wave height time series at the wave boundary locations > exactly the same format as the SFINCS bzsfile:

        0.0   4.570   5.570
    99999.0   4.570   5.570

'snapwave_btpfile = snapwave.btp' in sfincs.inp  > ascii file with peak wave periodtime series at the wave boundary locations > exactly the same format as the SFINCS bzsfile:

        0.0   10.0   12.0
    99999.0   11.0   13.0

'snapwave_bwdfile = snapwave.bwd' in sfincs.inp  > ascii file with wave direction time series at the wave boundary locations. direction in degrees, 'coming from wrt North'
 so 270 degrees is from the west, 0 degrees is from the North, 90 degrees is from the East, etc. > exactly the same format as the SFINCS bzsfile:

        0.0 270.000 90.000
    99999.0 280.000 100.000

'snapwave_bdsfile = snapwave.bds' in sfincs.inp  > ascii file with wave direction spreading in degrees series at the wave boundary locations > exactly the same format as the SFINCS bzsfile:

        0.0 30.000 20.000
    99999.0 30.000 20.000

In [ ]:
import numpy as np
import os

# --- Utility writer for 1 station ---
def write_snapwave_1point(path, time_sec, values):
    data = np.column_stack([time_sec, values])
    np.savetxt(path, data, fmt=["%.1f", "%.3f"])

# --- Boundary coordinate file (snapwave.bnd) ---
x = [324346.2]
y = [5041552.3]
xy = np.column_stack([x, y])
np.savetxt(os.path.join(sf_qt.root.path, "snapwave.bnd"), xy, fmt="%.3f")

# --- Wanted values ---
time_sec = [0, 99999.0]
bhs_1d = [3.570, 4.250]
btp_1d = [10.0, 11.0]
bwd_1d = [160.0, 150.0]
bds_1d = [30.0, 20.0]

# --- Write timeseries files into model_root ---
write_snapwave_1point(os.path.join(sf_qt.root.path, "snapwave.bhs"), time_sec, bhs_1d)
write_snapwave_1point(os.path.join(sf_qt.root.path, "snapwave.btp"), time_sec, btp_1d)
write_snapwave_1point(os.path.join(sf_qt.root.path, "snapwave.bwd"), time_sec, bwd_1d)
write_snapwave_1point(os.path.join(sf_qt.root.path, "snapwave.bds"), time_sec, bds_1d)

# To add to your sfincs.inp file afterwards
sf_qt.config.update(
**{
    "snapwave_bndfile" : r"snapwave.bnd",
    "snapwave_bhsfile" : r"snapwave.bhs",
    "snapwave_btpfile" : r"snapwave.btp",
    "snapwave_bwdfile" : r"snapwave.bwd",
    "snapwave_bdsfile" : r"snapwave.bds",

    }
)

### Add specific settings to include coupled SnapWave model

In [ ]:
sf_qt.config.update(
    **{
        "snapwave": 1, #=turn on snapwave
        "snapwave_igwaves" : 1, # IG balance off/on (for now start without)
        "snapwave_wind": 0, # to route wind forcing you supply to SFINCS into SnapWave, and allow wave growth due to wind. Don't turn on if you don't provide wind data
        "storefw": 1, # store some more wave related output variables if wanted
        "storewavdir": 1, # store wave direction output if wanted
        "dtwave": 1800, # coupling interval for SnapWave in seconds > as in Delft3Dflow-SWAN, you don't run the stationary wave spectral model continuously, but at intervals
        "snapwave_alpha": 1.0, # alpha parameter in Baldock wave breaking formula, default = 1.0
        "snapwave_gamma": 0.78, # gamma parameter in Baldock wave breaking formula, controls depth of wave breaking, default is 0.7
        "snapwave_hmin": 0.01, # minimum water depth for which cell SnapWave is calculated (can change during the simulation because of rising water levle)
        "snapwave_dtheta": 5, # direction bin size in degrees > use of 5 degrees is recommended
        "snapwave_fw": 0.02, # bottom friction factor for waves
        "snapwave_sector": 180, # wave direction sector > when not providing wind can generally be 180 degrees, otherwise set to 360 degrees
        "snapwave_niter": 100, #max number of iterations (is divided by 4 currently, for the 4 internal sweeps)
    }
)



<div style="border-left: 4px solid #960000ff; padding: 0.5em; background-color: #8ed6d69a;">
<b>⚠️ Note:</b> Note - besides not reusing an upwfile if the underlying quadtree grid has been changed, one should also not reuse if the directional grid has been changed!
So when running with 'dtheta = 10', the created upwfile cannot be used anymore if afterwards there is changed to 'dtheta=5' degrees (or vice versa) in sfincs.inp.
</div>

### Add wavemaker

For this you have to provide a MultiLineString geometry geojson file (can make in Qgis for instance).


<div style="border-left: 4px solid #960000ff; padding: 0.5em; background-color: #8ed6d69a;">
<b>⚠️ Note:</b> NOTE - clicking direction matters! Looking to the north, you should click from left to right, to have the waves be generated at the wavemaker in northward direction.

</div>

In [ ]:
filename = r'./wavemaker_example.geojson' #quadtree_grid_model_with_IG_v03

sf_qt.wave_makers.create(filename, merge=False)


### Inspect model

Inspect the model and forcing. Note, the variable used to describe the elevation in the model is now called "z", which is in line with the SFINCS kernel.

In [ ]:
sf_qt.plot_basemap(variable="z", plot_region=True, plot_bounds=False, bmap="sat", zoomlevel=12)
sf_qt.plot_forcing()

### Write the model

When writing away a quadtree model, you will see a different folder structure for in your SFINCS root. A single Quadtree NetCDF is created, this NetCDF replaces the old files that included infromation about the domain such that:

**Regular**

* sfincs.dep      
* sfincs.ind     
* sfincs.msk        
* sfincs.man  

**Quadtree**

* sfincs.nc

This reduces the amount of files that is created in your model folder.

In [ ]:
sf_qt.write()

<div style="border-left: 4px solid #960000ff; padding: 0.5em; background-color: #ff4d4df6;">
<b>⚠️ Note:</b> You might get some "INFO" statement on missing quadtree_infiltration variables. This makes sense since we did not create these layers yet, and this is also still work in progress.

</div>


## Run and read results, make plots and animation

When running the model, the log file should show that the quadtree refinement are activated. If not, something went wrong. Remember to change around the executables path to the location of your own sfincs.exe!

In [ ]:
from hydromt_sfincs.run import run_sfincs

sfincs_exe = r"..\..\..\..\executables\SFINCS_2025_02_release\SFINCS_v2.3.0_mt_Faber_release_exe\sfincs.exe"

run_sfincs(model_root=sf_qt.root.path, sfincs_exe=sfincs_exe)

<div style="border-left: 4px solid #960000ff; padding: 0.5em; background-color: #8ed6d69a;">
<b>⚠️ Note:</b> Check in you Task manager whether SFINCS is running with 100% parallelization or not ! It might be that when running through the notebook, or when a lot of other programs are active, SFINCS is not running as efficiently as possible. 

If so, you can always just run SFINCS using a separate batchfile (https://sfincs.readthedocs.io/en/latest/example.html#using-batch-file).

</div>

### Results

For the results, nothing to much changes if you use the build in scripts of hydromt. HydroMT recognizes the model folder structure and the Quadtree type of input and output. Under the hood, it uses **xugrid**, to open quadtree type of files.

In [ ]:
# read results
model_root = Path(model_root) # String to Path conversion, because HydroMT-SFINCS expects a Path object, needs to be fixed 
sf_qt.root.set(path=model_root,mode="r") #reading mode
sf_qt.output.read(fn_map="sfincs_map.nc", fn_his="noexistent.nc")

NOTE - you now get extra output variables in sfincs_map.nc because of running including SnapWave:
- snapwavemsk 
    - Your snapwave mask on the quadtree mesh, can be different than your SFINCS mask if wanted
- hm0
    - Significant wave height of short waves
- hm0ig
    - Significant wave height of incoming infragravity waves, all zeros if SnapWave is run without igwaves turned on

Optional:
- wavdir
    - Wave direction in degrees, if storewavdir = 1 in sfincs.inp
- tp
    - Peak wave period of short waves, if storefw = 1 in sfincs.inp
- tpig
    - Peak wave period of infragravity waves, all zeros if SnapWave is run without igwaves turned on, if storefw = 1 in sfincs.inp
- snapwavedepth
    - Interpolated water depth from SFINCS to SnapWave cells, can change over time due to changing water levels, if storefw = 1 in sfincs.inp    
- fwx&fwy
    - Wave forces in x&y-direction, this is what is routed from SnapWave to SFINCS to resolve short wave-induced setup, if storefw = 1 in sfincs.inp
- beta
    - Directionally_averaged_local_bed_slope, if storefw = 1 in sfincs.inp

In [ ]:
sf_qt.output.data.keys()

<div style="border-left: 4px solid #960000ff; padding: 0.5em; background-color: #8ed6d69a;">
<b>⚠️ Note:</b> In sfincs_his.nc you now also get extra output variables because of running including SnapWave, in case observation points are provided.
However, some wave related parameters in the sfincs_his.nc file have the same name still as in sfincs_map.nc (e.g. hm0, hm0ig, tp, tpig), causing them not to be added to the 'sf_wt.output.data' list as they are double. But, when loading in the sfincs_his.nc file separately using xarray, these outputs are available to you (see example below). Variable names might be changed in the future to overcome this.

</div>


In [ ]:
# now plot the bed levels
zb = sf_qt.output.data["zb"]
sf_qt.plot_basemap(variable=zb, plot_bounds=False, bmap="sat", cmap="terrain", vmax=5, zoomlevel=12)

In [ ]:
zsmax = sf_qt.output.data["zsmax"].max(dim="timemax")
sf_qt.plot_basemap(variable=zsmax, plot_bounds=False, bmap="sat", cmap="viridis", zoomlevel=12)

In [ ]:
# plot rough estimation of water depth
h = zsmax - zb
# only plot where water depth > 0.1
h = h.where(h > 0.1)
# add label to h
h.name = "Water depth"

sf_qt.plot_basemap(variable=h, plot_bounds=False, bmap="sat", cmap="viridis", zoomlevel=12)

When you start plotting the results yourself, you should use **.ugrid** to plot you results, for more information about Xugrid, check out the API: [xugrid read the docs](https://deltares.github.io/xugrid/index.html)

In [ ]:
# create zs plot and save to mod.root/figs/sfincs_zs.mp4
# requires ffmpeg install with "conda install ffmpeg -c conda-forge"
import matplotlib.pyplot as plt
import numpy as np
from matplotlib import animation

step = 1  # one frame every <step> dtout
cbar_kwargs = {"shrink": 0.6, "anchor": (0, 0)}
da_zs = sf_qt.output.data["zs"]

def update_plot(i, da_zs, cax_zs):
    da_zsi = da_zs.isel(time=i)
    t = da_zsi.time.dt.strftime("%d-%B-%Y %H:%M:%S").item()
    ax.set_title(f"SFINCS water level {t}")
    cax_zs.set_array(da_zsi.values.ravel())


fig, ax = sf_qt.plot_basemap(
    ax=ax, variable="", plot_bounds=False, bmap="sat", zoomlevel=13, figsize=(11, 7)
)
cax_zs = da_zs.isel(time=0).ugrid.plot(
    ax=ax, vmin=0, vmax=1, cmap=plt.cm.viridis, zorder=1, cbar_kwargs=cbar_kwargs # Using UGRID to plot!
)
plt.close()  # to prevent double plot

ani = animation.FuncAnimation(
    fig,
    update_plot,
    frames=np.arange(0, da_zs.time.size, step),
    interval=250,  # ms between frames
    fargs=(
        da_zs,
        cax_zs,
    ),
)

# to save to mp4
# ani.save(join(mod.root, 'figs', 'sfincs_h.mp4'), fps=4, dpi=200)

# to show in notebook:
from IPython.display import HTML

HTML(ani.to_html5_video())

### Plotting SnapWave results

Significant wave height:

In [ ]:
hm0 = sf_qt.output.data["hm0"].isel(time=-1) #last timestep in this case
hm0 = hm0.where(hm0 > 0.1)
sf_qt.plot_basemap(variable=hm0, plot_bounds=False, bmap="sat", cmap="viridis", zoomlevel=12)

Peak wave period:

In [ ]:
tp = sf_qt.output.data["tp"].isel(time=-1) #last timestep in this case
tp = tp.where(hm0 > 0.1)
sf_qt.plot_basemap(variable=tp, plot_bounds=False, bmap="sat", cmap="viridis", zoomlevel=12)

<div style="border-left: 4px solid #960000ff; padding: 0.5em; background-color: #8ed6d69a;">
<b>⚠️ Note:</b> Note that the wave period is spatially uniform in case no wind growth is included in SnapWave.

</div>

Wave direction:

In [ ]:
wavdir = sf_qt.output.data["wavdir"].isel(time=-1) #last timestep in this case
wavdir = wavdir.where(hm0 > 0.1)
sf_qt.plot_basemap(variable=wavdir, plot_bounds=False, bmap="sat", cmap="viridis", zoomlevel=12)

Interpolated water depth in Snapwave in time (can very if tide/storm surge changes in SFINCS part):

In [ ]:
snapwavedepth = sf_qt.output.data["snapwavedepth"].isel(time=-1) #last timestep in this case
snapwavedepth = snapwavedepth.where(hm0 > 0.1)
sf_qt.plot_basemap(variable=snapwavedepth, plot_bounds=False, bmap="sat", cmap="viridis", zoomlevel=12)

IG wave height:

In [ ]:
hm0ig = sf_qt.output.data["hm0ig"].isel(time=-1) #last timestep in this case
hm0ig = hm0ig.where(hm0 > 0.1)
sf_qt.plot_basemap(variable=hm0ig, plot_bounds=False, bmap="sat", cmap="viridis", zoomlevel=12)

In [ ]:
# create zs plot and save to mod.root/figs/sfincs_zs.mp4
# requires ffmpeg install with "conda install ffmpeg -c conda-forge"
import matplotlib.pyplot as plt
import numpy as np
from matplotlib import animation

step = 1  # one frame every <step> dtout
cbar_kwargs = {"shrink": 0.6, "anchor": (0, 0)}
da_hm0 = sf_qt.output.data["hm0"]
da_hm0 = da_hm0.where(da_hm0 > 0.0)

def update_plot(i, da_hm0, cax_hm0):
    da_hm0i = da_hm0.isel(time=i)
    t = da_hm0i.time.dt.strftime("%d-%B-%Y %H:%M:%S").item()
    ax.set_title(f"SnapWave Hm0 {t}")
    cax_hm0.set_array(da_hm0i.values.ravel())


fig, ax = sf_qt.plot_basemap(
    ax=ax, variable="", plot_bounds=False, bmap="sat", zoomlevel=13, figsize=(11, 7)
)
cax_hm0 = da_hm0.isel(time=0).ugrid.plot(
    ax=ax, vmin=0, vmax=4, cmap=plt.cm.viridis, zorder=1, cbar_kwargs=cbar_kwargs # Using UGRID to plot!
)
plt.close()  # to prevent double plot

ani = animation.FuncAnimation(
    fig,
    update_plot,
    frames=np.arange(0, da_hm0.time.size, step),
    interval=250,  # ms between frames
    fargs=(
        da_hm0,
        cax_hm0,
    ),
)

# to save to mp4
# ani.save(join(mod.root, 'figs', 'sfincs_h.mp4'), fps=4, dpi=200)

# to show in notebook:
from IPython.display import HTML

HTML(ani.to_html5_video())

In [ ]:
# create zs plot and save to mod.root/figs/sfincs_zs.mp4
# requires ffmpeg install with "conda install ffmpeg -c conda-forge"
import matplotlib.pyplot as plt
import numpy as np
from matplotlib import animation

step = 1  # one frame every <step> dtout
cbar_kwargs = {"shrink": 0.6, "anchor": (0, 0)}
da_hm0 = sf_qt.output.data["hm0ig"]
da_hm0 = da_hm0.where(da_hm0 > 0.0)

def update_plot(i, da_hm0, cax_hm0):
    da_hm0i = da_hm0.isel(time=i)
    t = da_hm0i.time.dt.strftime("%d-%B-%Y %H:%M:%S").item()
    ax.set_title(f"SnapWave Hm0 IG {t}")
    cax_hm0.set_array(da_hm0i.values.ravel())


fig, ax = sf_qt.plot_basemap(
    ax=ax, variable="", plot_bounds=False, bmap="sat", zoomlevel=13, figsize=(11, 7)
)
cax_hm0 = da_hm0.isel(time=0).ugrid.plot(
    ax=ax, vmin=0, vmax=1.0, cmap=plt.cm.viridis, zorder=1, cbar_kwargs=cbar_kwargs # Using UGRID to plot!
)
plt.close()  # to prevent double plot

ani = animation.FuncAnimation(
    fig,
    update_plot,
    frames=np.arange(0, da_hm0.time.size, step),
    interval=250,  # ms between frames
    fargs=(
        da_hm0,
        cax_hm0,
    ),
)

# to save to mp4
# ani.save(join(mod.root, 'figs', 'sfincs_h.mp4'), fps=4, dpi=200)

# to show in notebook:
from IPython.display import HTML

HTML(ani.to_html5_video())

#### Example of how to plot a transect, from this unstructured quadtree grid using xugrid:

In [ ]:
da_hm0 = hm0.ugrid.intersect_line(start=(324033, 5043508), end=(322560, 5045533))

# so now da_hm0 is a slice along this transect line
da_hm0

In [ ]:
# add the elevation as well
da_zb = zb.ugrid.intersect_line(start=(324033, 5043508), end=(322560, 5045533))
da_zb.plot(x="mesh2d_s")

# now plot
da_hm0.plot(x="mesh2d_s") # 'mesh2d_s' is the distance along the transect

# with IG
da_hm0ig = hm0ig.ugrid.intersect_line(start=(324033, 5043508), end=(322560, 5045533))
da_hm0ig.plot(x="mesh2d_s")

In [ ]:
# plot elevation and water level:
da_zb.plot(x="mesh2d_s")

da_zsmax = zsmax.ugrid.intersect_line(start=(324033, 5043508), end=(322560, 5045533))
da_zsmax.plot(x="mesh2d_s")

In [ ]:
# create zs plot and save to mod.root/figs/sfincs_zs.mp4
# requires ffmpeg install with "conda install ffmpeg -c conda-forge"
import matplotlib.pyplot as plt
import numpy as np
from matplotlib import animation

step = 1  # one frame every <step> dtout
cbar_kwargs = {"shrink": 0.6, "anchor": (0, 0)}
da_zs = sf_qt.output.data["zs"]
# da_hm0 = da_hm0.where(da_hm0 > 0.0)



def update_plot(i, da_zs, cax_zs):
    da_zsi = da_zs.isel(time=i).ugrid.intersect_line(start=(324033, 5043508), end=(322560, 5045533))
    t = da_zsi.time.dt.strftime("%d-%B-%Y %H:%M:%S").item()
    ax.set_title(f"SFINCS zs {t}")
    # cax_zs.set_array(da_zsi.values.ravel())
    # cax_zs = da_zsi.plot(
    #     ax=ax, x="mesh2d_s",zorder=1 # Using UGRID to plot!
    # )    
    # cax_zs.set_ydata(da_zsi)
    cax_zs.remove()
    da_zsi.plot(ax=ax, x="mesh2d_s")

# fig, ax = sf_qt.plot_basemap(
#     ax=ax, variable="", plot_bounds=False, bmap="sat", zoomlevel=13, figsize=(11, 7)
# )
fig, ax = plt.subplots(
    figsize=(11, 7)
)
da_zb = zb.ugrid.intersect_line(start=(324033, 5043508), end=(322560, 5045533))
da_zb.plot(ax=ax, x="mesh2d_s")

cax_zs = da_zs.isel(time=0).ugrid.intersect_line(start=(324033, 5043508), end=(322560, 5045533)).plot(
    ax=ax, x="mesh2d_s")#,zorder=1 # Using UGRID to plot!
# )
plt.close()  # to prevent double plot

ani = animation.FuncAnimation(
    fig,
    update_plot,
    frames=np.arange(0, da_zs.time.size, step),
    interval=250,  # ms between frames
    fargs=(
        da_zs,
        cax_zs,
    ),
)

# to save to mp4
# ani.save(join(mod.root, 'figs', 'sfincs_zs.mp4'), fps=4, dpi=200)

# to show in notebook:
from IPython.display import HTML

HTML(ani.to_html5_video())

In [ ]:
cax_zs

Plot time-series of wave conditions

In [ ]:
import xarray as xr

dahis = xr.open_dataset(os.path.join(sf_qt.root.path, "sfincs_his.nc"))
dahis.data_vars.keys()

Plot significant wave height

In [ ]:
dahis.hm0.plot()

<div style="border-left: 4px solid #960000ff; padding: 0.5em; background-color: #8ed6d69a;">
<b>⚠️ Note:</b> Wave conditions from SnapWave are constant for the period of 'dtwave' (here 1800 seconds) since it is run in a stationary mode. Values in this case are written away every 'dthisout' seconds (here 600). So the values at t=00:00, t=00:10 and t=00:20 are the same, then at t=00:30 SnapWave is updated and the new computed values are written away at t=00:30, t=00:40 and t=00:50 before SnapWave is updated again at t=01:00.
Thus note that the update interval of SnapWave, and the output interval are not the same (also for sfincs_map.nc).

</div>


Plot water level at observation point

In [ ]:
dahis.point_zs.plot()

<div style="border-left: 4px solid #960000ff; padding: 0.5em; background-color: #8ed6d69a;">
<b>⚠️ Note:</b> Note that the SnapWave solver updates every 'dtwave' (here 1800 seconds), using the water level of that exact moment. So the water levels provided to SnapWave are updated at t=00:00, t=00:30, t=01:00 etc. One can see that the water level has some initial wiggles because 'zsini = 0.25' is not equal to the initial water level in the bzsfile (0.425   0.426   0.428
), this results in an incoming and reflecting wave. The water level at t=00:30 is higher because of this, and this on it's part then influences the shoaling and breaking of the significant wave height in the figure above. Message is to always be aware of model spinup (and try to overcome), as these influence your modelling results.

</div>

Plot Tp

In [ ]:
dahis.tp.plot()

<div style="border-left: 4px solid #960000ff; padding: 0.5em; background-color: #8ed6d69a;">
<b>⚠️ Note:</b> Note that the wave period is not influence by the water level, and therefore no effect of the wiggles in the water level have an effect here.

</div>

In [ ]:
dahis.hm0ig.plot()

In [ ]:
dahis.tpig.plot()